In [1]:
melody_file_path = "input/mel/001.mid"

accompaniment_file_path = "acc-poly-pattern_resampled/c_major_poly_stride.mid"

In [2]:
from app.acc_transpose.music21_key_detect import detect_key

tonic, mode, confidence = detect_key(melody_file_path)
print(f"Detected key: {tonic} ,mode: {mode} (confidence: {confidence:.2f})")

Detected key: Eb ,mode: minor (confidence: 0.70)


In [3]:
# from app.acc_transpose.transpose_c_major import transpose_pm, parse_keys
# from app.acc_transpose.transpose_key import get_semitones_for_transposition, transpose_pm, transpose_midi_file
from app.acc_transpose.transpose_c_major import transpose_midi_file

transpose_midi_file(accompaniment_file_path, ["EBm"], "transposed")

3


/home/ubuntu/ugrip/yuanhsin/streamMuse2/.venv/lib/python3.10/site-packages/pretty_midi/instrument.py:11: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


[PosixPath('/home/ubuntu/ugrip/yuanhsin/streamMuse2/transposed/c_major_poly_stride_to_Eflat_minor.mid')]

In [4]:
from app.acc_transpose.transpose_c_major2 import transpose_midi_file

transpose_midi_file(accompaniment_file_path, ["EBm"], "transposed2")

[PosixPath('/home/ubuntu/ugrip/yuanhsin/streamMuse2/transposed2/c_major_poly_stride_to_Eflat_minor.mid')]

In [5]:
from app.acc_transpose.music21_key_detect import detect_key

tonic, mode, confidence = detect_key(accompaniment_file_path)
print(f"Detected key: {tonic} ,mode: {mode} (confidence: {confidence:.2f})")


Detected key: C ,mode: major (confidence: 0.89)


In [6]:
from app.acc_transpose.music21_key_detect import detect_key

tonic, mode, confidence = detect_key("transposed/c_major_poly_stride_to_Eflat_minor.mid")
print(f"Detected key: {tonic} ,mode: {mode} (confidence: {confidence:.2f})")


Detected key: Eb ,mode: major (confidence: 0.89)


In [7]:
from app.acc_transpose.music21_key_detect import detect_key

tonic, mode, confidence = detect_key("transposed2/c_major_poly_stride_to_Eflat_minor.mid")
print(f"Detected key: {tonic} ,mode: {mode} (confidence: {confidence:.2f})")


Detected key: F# ,mode: major (confidence: 0.92)


In [ ]:
# ...existing code...
import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set(style="whitegrid")


# 把一个向量（最后一维）当作一个 token 来统计
def token_counts_from_tensor_vectors(t):
    """
    t: torch.Tensor, 最后一个维度是一个向量 token，例如 shape [N, L] 或 [B, T, L]
    返回 (counts, labels)
      - counts: np.array of counts for each unique vector token
      - labels: list of tuple/int sequence representing the token
    """
    if t.dim() == 0:
        return np.array([]), []
    # 把所有样本 flatten 成 (M, L)
    L = t.shape[-1]
    arr = t.reshape(-1, L).cpu().numpy().astype(np.int64)
    uniques, counts = np.unique(arr, axis=0, return_counts=True)
    # 用 tuple 作为 token label（可换成 string）
    labels = [tuple(row.tolist()) for row in uniques]
    return counts, labels


# 兼容老接口：单值 token 仍旧使用原来的统计方式
def token_counts_from_tensor(t):
    if t.dim() == 1 or (t.dim() > 1 and t.shape[-1] == 1):
        tokens = t.view(-1).cpu().numpy().astype(np.int64)
        max_tok = int(tokens.max()) + 1 if tokens.size else 0
        counts = np.bincount(tokens, minlength=max_tok)
        labels = [i for i in range(len(counts))]
        return counts, labels
    # 否则视为向量 token
    return token_counts_from_tensor_vectors(t)


def group_top_n_with_labels(counts, labels, n):
    total = counts.sum()
    order = np.argsort(-counts)
    top = order[:n]
    others = order[n:]
    groups = [labels[i] for i in top]
    pcts = [counts[i] / total for i in top]
    groups.append("others")
    pcts.append(counts[others].sum() / total)
    return groups, pcts


def merge_small_until_threshold_with_labels(counts, labels, threshold=0.05):
    total = counts.sum()
    order = np.argsort(counts)  # asc
    merged = []
    merged_pct = 0.0
    idx = 0
    while idx < len(order) and merged_pct < threshold:
        merged.append(order[idx])
        merged_pct += counts[order[idx]] / total
        idx += 1
    rest = order[idx:]
    rest_desc = rest[np.argsort(-counts[rest])]
    groups = [labels[i] for i in rest_desc]
    pcts = [counts[i] / total for i in rest_desc]
    if merged:
        groups.append("merged_small")
        pcts.append(sum(counts[i] for i in merged) / total)
    return groups, pcts


def plot_cumulative(counts, labels=None, top_k=50, threshold=0.05, title="Token distribution"):
    total = counts.sum()
    order = np.argsort(-counts)
    freqs = counts[order] / total
    # labels sorted
    sorted_labels = [labels[i] for i in order] if labels is not None else None

    cum = np.cumsum(freqs)
    x = np.arange(len(freqs))
    plt.figure(figsize=(10, 5))
    plt.plot(x[:top_k], cum[:top_k], marker="o")
    plt.axhline(threshold, color="red", linestyle="--", label=f"{threshold * 100:.0f}% threshold")
    plt.xlabel("tokens (ranked)")
    plt.ylabel("cumulative percentage")
    plt.title(title + " - cumulative (top %d)" % top_k)
    plt.legend()
    plt.grid(True)
    plt.show()

    # 条形图（前 top_k）
    plt.figure(figsize=(14, 6))
    top_freqs = freqs[:top_k]
    # 显示标签（如果是 tuple 形式会被转换成字符串）
    xticks = [str(sorted_labels[i]) if sorted_labels is not None else f"#{i + 1}" for i in range(top_k)]
    sns.barplot(x=xticks, y=top_freqs)
    plt.xticks(rotation=90)
    plt.title(title + " - top %d tokens frequency" % top_k)
    plt.ylabel("fraction")
    plt.show()


# 示例：对 mel 做统计并画图（把每个 12 维向量视作 token）
mel = torch.load("data/pop909_mel_cp4.pt")
acc = torch.load("data/pop909_acc_cp4.pt")

print("mel shape:", mel.shape, "acc shape:", acc.shape)

mel_counts, mel_labels = token_counts_from_tensor(mel)
acc_counts, acc_labels = token_counts_from_tensor(acc)

print("unique mel tokens:", len(mel_counts), "unique acc tokens:", len(acc_counts))

# Top 20 groups（示例）
g1, p1 = group_top_n_with_labels(mel_counts, mel_labels, 20)
print("Top-20 groups (mel):", list(zip(g1, p1))[:5], "...")

# 合并小类（示例 threshold=5%）
g2, p2 = merge_small_until_threshold_with_labels(mel_counts, mel_labels, threshold=0.05)
print("Merged groups (mel):", list(zip(g2, p2))[:10], "...")

# 画累计分布（带 labels）
plot_cumulative(mel_counts, labels=mel_labels, top_k=100, threshold=0.05, title="Mel tokens (vector-based)")
plot_cumulative(acc_counts, labels=acc_labels, top_k=100, threshold=0.05, title="Acc tokens (vector-based)")
# ...existing code...
